### Inferring single cell trajectories and performing cell type composition analysis - session 3 (10/03/25)

Elise Smith

### Part 1 - pseudotime analysis

- In this session we are firstly aiming to infer trajectories of cells from the bone marrow, using scRNA-seq data
- We will use the method diffusion pseudotime (DPT, https://www.nature.com/articles/nmeth.3971)

### Background

Tutorial based on https://www.sc-best-practices.org/trajectories/pseudotemporal.html

Motivation:
- We are interested in understanding the lineage relationships between cells, for example the order of cell types during the process of differentiation, including intermediate states in this process
- How can we infer these relationships between single cells based on their RNA, without time series data? 


- Understanding the dynamic processes underlying lineage commitment is challenging using traditional single-cell methods
- Cells are destroyed during sequencing for most scRNA-seq protocols, therefore, we can't track their development by looking at changes in gene expression profiles over time
- In most cases, dynamics of a biological process need to be inferred from 'snapshots' of this process

Trajectory analysis:
- We can use a single sample containing cells that are all at slightly different stages along a path of differentiation
- Within this sample, some cells are more differentiated than others, although they are all captured at the same time
- This is applicable to various biological processes, including cancer progression and healthy development
- Trajectory inference is particularly useful in these scenarios, since *in vivo* lineage tracing is not easily applied




In this tutorial, we will use a dataset originally from Setty et al., 2019 (Nature Biotechnology, https://doi.org/10.1038/s41587-019-0068-4)
- In this study, they investigated hematopoietic differentiation
- Sequenced bone marrow stem and progenitor cells from healthy donors, enriched for CD34 (stem/progenitor cell marker)
- Performed scRNA-seq with 10x genomics sequencing

Haematopoeisis:
- Development of mature blood cells from haematopoietic stem cells (HSCs)
- HSCs develop into myeloid or lymphoid progenitors and eventually form differentiated cells, such as lymphoid (T/B cells), erythroid and myeloid subsets 
- This is a good model for trajectory analysis since 'a single bone marrow sample contains the full spectrum of cell states in hematopoiesis and importantly the frequencies of each cell state'



Trajectory/pseudotime methods:
- Rank cells relative to each other according to their position along a trajectory, assigning a pseudotime value to each cell, representing where the cell is along the path
- Mature cells are assigned larger values, compared to less differentiated cells that are assigned smaller values
- In this example, HSCs are assigned a low pseudotime, whilst differentiated cells e.g. erythroid cells are assigned a high pseudotime
- These values are based on the transcriptomic profile of the cell when using scRNA-seq data


- There are multiple different methods for pseudotime construction
- Summarised here https://www.sc-best-practices.org/trajectories/pseudotemporal.html#pseudotime-construction
- An assumption of pseudotime algorithms is that differentiation is unidirectional and proceeds toward functionally mature cells, which may not always be the case (cancer/tissue regeneration)
- Therefore, these methods may work better e.g. for haematopoesis than in other situations

Diffusion pseudotime construction:
- We will use the method diffusion pseudotime (DPT) from Haghverdi et al., 2016
- This is based on a probabilistic framework, assigning transition probabilities to ordered cell-cell pairs
- This method uses diffusion maps - a non-linear dimensionality reduction method 
- These are useful for representing the data since we wouldn't expect discrete clusters in the data, but instead continuous branching lineages of cells
- 'DPT calculates the probability of cells transitioning into each other using random walks from a user-provided root cell and uses the differences between these probabilities as pseudotime'


### Performing trajectory analysis

In [ ]:
from pathlib import Path
import scanpy as sc
import numpy as np
import pandas as pd

sc.settings.set_figure_params(scanpy = True, dpi=100, transparent=True, vector_friendly = True, frameon = False) 

#### Load data - stem and progenitor CD34+ cells from the bone marrow of healthy donors

In [ ]:
adata = sc.read(
    filename= 'bone_marrow.h5ad',
    backup_url= "https://figshare.com/ndownloader/files/35826944")

Check how many cells and genes there are in the dataset


In [ ]:
adata.shape

### Check QC metrics - number of genes and counts, % mt and ribosomal reads

These are the same steps that were used in the previous two tutorials

Which types of genes are the most highly expressed in the dataset?

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20)


Calculate QC metrics

In [ ]:
# mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))

# compute QC metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo"], inplace=True, log1p=False, percent_top=None)

What is the average number of genes and % of reads mapping to mitochondrial and ribosomal genes?

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo'],
             jitter=0, multi_panel=True)

Filter out genes with few counts - how many genes were removed?

Genes expressed in few cells/with few counts are generally less informative so we can remove these. However, this may not always be the case, e.g. for rare cell populations

You can also try different QC cut-offs


In [ ]:
print(adata.shape)
sc.pp.filter_genes(adata, min_counts=20)
print(adata.shape)

We will remove ribosomal and mitochondrial genes that are often associated with data quality

Removal of these genes also aids marker gene identification

In [ ]:
print(adata.shape)
adata = adata[:, ~adata.var["mt"].values]
adata = adata[:, ~adata.var["ribo"].values].copy()
print(adata.shape)


Check what the minimum number of genes per cell is

In this case each cell has over 300 unique genes detected, so we will not perform any further filtering


In [ ]:
np.min(adata.obs['n_genes_by_counts'])

### Data normalisation and clustering


Now we will normalise the data, and log1p transform counts to reduce the effect of outliers

Then we will find highly variable genes, which reduces the dimensionality of the dataset and finds the most informative genes


Genes varying the most across the dataset are likely to include the most biologically informative genes and are used in subsequent analysis steps e.g. PCA

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata)


Next we will run PCA, that reduces the dimensionality of the data, revealing the main axes of variation and denoising the data

We will then compute the neighbourhood graph which using the PCA representation of the data matrix

Here we will use 10 principal components which is the number used in the tutorial

You could also try with other numbers of principal components

In [ ]:
sc.tl.pca(adata)
sc.pp.neighbors(adata, n_pcs=10)

We will then embed the neighbourhood graph using UMAP and visualise the data using the two-dimensional UMAP representation


First we will visualise some QC metrics

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata,  color=["n_genes_by_counts", "pct_counts_mt", "pct_counts_ribo"], vmax = [3000,10,50], cmap = 'plasma_r')


### Marker genes 

Now let's check some marker genes

In [ ]:
import itertools
markers = {'Stem/progenitor cells':['CD34', 'CD52'],
           'Early myeloid':['MPO'],
           'Lymphoid':['CD79B'],
           'Erythroid':['GATA1', 'KLF1'],
           'DC':['IGKC','JCHAIN','IRF8'],
           'Megakaryocyte':['ITGA2B'], # gene for CD41
           'Monocyte':['S100A8','AZU1','LYZ']}

markers = list(itertools.chain(*list(markers.values())))  # get all markers in a single list
sc.pl.umap(adata, color = markers, color_map = 'plasma_r')

We'll also look at marker gene expression using a tSNE representation of the data
This also lets us look at the scRNA-seq data in a 2D representation 
In some situations UMAP or tSNE may be better than the other

Which representation looks better for this data?

In [ ]:
sc.pl.tsne(adata, color = markers, color_map = 'plasma_r')

Both tSNE and UMAP representations show separation of MPO+ (early myeloid), CD79B+ (lymphoid) and ITGA2B+ (megakaryocyte) cells from the other cells

You can try plotting additional genes using the following code

In [ ]:
genes = ['MPO','CD52'] # change to genes that you would like to plot
sc.pl.umap(adata, color = genes, cmap = 'plasma_r')

### Differential expression analysis to identify clusters

One helpful method for annotating is to look at upregulated genes in each cluster using differential expression analysis

First we will compute clusters with a few different values for the resolution parameter, you can also try different ones

In [ ]:
sc.tl.leiden(adata, resolution=0.6, key_added = 'clusters_r06')
sc.tl.leiden(adata, resolution=0.4, key_added = 'clusters_r04')
sc.tl.leiden(adata, resolution=0.2, key_added = 'clusters_r02')

In [ ]:
sc.pl.umap(adata, color = ['clusters_r02','clusters_r04','clusters_r06'])

You can change the number of genes that are output by altering .head()

Searching for gene names in publications or databases can help determine the cell type that each cluster corresponds to

Try plotting genes using the code above to help identify clusters

In [ ]:
sc.tl.rank_genes_groups(adata, "clusters_r02", max_iter=1000, key_added = 'cluster_DEGs') # change clusters_r02 if you used a different resolution
pd.DataFrame(adata.uns["cluster_DEGs"]["names"]).head(30)

We can also create a dotplot using a dictionary of genes above

What do the numbers on the left side of the plot correspond to?


In [ ]:
markers = {'Stem/progenitor cells':['CD34', 'CD52'],
           'Early myeloid':['MPO'],
           'Lymphoid':['CD79B'],
           'Erythroid':['GATA1', 'KLF1'],
           'DC':['IGKC','JCHAIN','IRF8'],
           'Megakaryocyte':['ITGA2B'], # gene for CD41
           'Monocyte':['S100A8','AZU1','LYZ']}
sc.pl.dotplot(adata, var_names = markers, color_map = 'plasma_r', groupby = 'clusters_r02', standard_scale = 'var')

### Cluster labels - rough annotation


Add in cluster labels using your annotations next to the numbers, here they're currently named 'cell_type'

Fill in the dictionary below and visualise the clusters. At this stage, it's just a rough annotation


In [ ]:
# Add in extra lines depending on how many clusters you have
# If you have any unlabelled clusters, you can label these as 'unknown'

adata.obs['cell_type'] = (
    adata.obs['clusters_r02'] # change this if you use a different resolution, or your naming is different
    .map(lambda x: {'0':'', # example: {'0': 'erythrocyte',
                    '1':'',
                    '2':'',
                    '3': '',
                    '4': '',
                    '5': '',
                    '6': ''}.get(x, x)).astype("category"))


In [ ]:
sc.pl.umap(adata, color = 'cell_type')


Now we can visualise the cell subpopulations using labels that were previously annotated

Abbreviations:
- HSC - haematopoeitic stem cell
- Ery - erythrocyte
- Mono - monocyte
- CLP - common lymphoid progenitor 
- Mega - megakaryote
- DC - dendritic cell

Which representation looks better?


In [ ]:
import warnings
warnings.filterwarnings('ignore')

sc.pl.umap(adata, color = 'clusters')
sc.pl.tsne(adata, color = 'clusters')

tSNE seems to show more distinct separation for some of the populations, so will continue with tSNE for now

Re-check the dotplot using the provided annotations

You can re-order the dictionary to aid visualisation

In [ ]:
warnings.filterwarnings('default')

markers = {'Stem/progenitor cells':['CD34', 'CD52'],
           'Early myeloid':['MPO'],
           'Lymphoid':['CD79B'],
           'Erythroid':['GATA1', 'KLF1'],
           'DC':['IGKC','JCHAIN','IRF8'],
           'Megakaryocyte':['ITGA2B'], # gene for CD41
           'Monocyte':['S100A8','AZU1','LYZ']}
sc.pl.dotplot(adata, var_names = markers, color_map = 'plasma_r', groupby = 'clusters', standard_scale = 'var')

### Pseudotime analysis

First we calculate the diffusion maps required for DPT

In [ ]:
sc.tl.diffmap(adata)

Now let's look at the diffusion components - similar to looking at principal components after PCA
- This will help us find a possible 'root' cell for our analysis
- The root cell is an initial cell from which our trajectory begins 
- Which cell type would the trajectory start from in haematopoeisis?

In [ ]:
sc.pl.scatter(
    adata, basis="diffmap", color=["clusters"], components=[1,2],
    frameon = True, title = 'Diffusion components 1 and 2')

sc.pl.scatter(
    adata, basis="diffmap", color=["clusters"], components=[2,3],
    frameon = True, title = 'Diffusion components 2 and 3')

sc.pl.scatter(
    adata, basis="diffmap", color=["clusters"], components=[3,4],
    frameon = True, title = 'Diffusion components 3 and 4')

- Plotting DC2 and DC3 shows that haematopoeitic stem cells are towards the extreme of DC3
- We know that haematopoeitic stem cells represent the start of the trajectory, although we don't know  which specific cell within the HSCs is the initial cell
- We will take the stem cell with the most extreme diffusion component in DC3 as the root cell

In [ ]:
root_ixs = adata.obsm["X_diffmap"][:, 3].argmin()
adata.uns["iroot"] = root_ixs

Run pseudotime analysis 

In [ ]:
sc.tl.dpt(adata)

Plot the DPT pseudotime values alongside the pre-annotated clusters

We will use the tSNE representation to look at the results from DPT

- Does this correspond to the expected outcome based on the biology we know of haematopoeisis?



In [ ]:
sc.pl.scatter(adata, basis="tsne",
    color=["dpt_pseudotime", "clusters"], color_map="plasma_r")

As mentioned before, multiple methods of pseudotime inference exist

We will now compare the DPT results to pre-computed results from the method Palantir

- What are the differences between the results from these two methods?

In [ ]:
sc.pl.tsne(adata, color=["dpt_pseudotime", "palantir_pseudotime", "clusters"], 
           color_map="plasma_r", ncols = 2)

We can also use a violin plot to compare the results from the two pseudotime methods

- Does this suggest that one method is more biologically accurate?

In [ ]:
sc.settings.set_figure_params(scanpy = True, dpi=80, transparent=True, vector_friendly = True, frameon = False, fontsize = 10) 
sc.pl.violin(
    adata, keys=["dpt_pseudotime", "palantir_pseudotime"],
    groupby="clusters", rotation=45,
    order=[
        "HSC_1",
        "HSC_2",
        "Precursors",
        "Ery_1",
        "Ery_2",
        "Mono_1",
        "Mono_2",
        "CLP",
        "DCs",
        "Mega"])

Extension: Try 'Trajectory inference for hematopoiesis in mouse' tutorial https://scanpy-tutorials.readthedocs.io/en/latest/paga-paul15.html

## Part 2 - Cell type composition analysis

Based on tutorial from https://www.sc-best-practices.org/conditions/compositional.html#compositional-analysis

See additional sections in the above tutorial for more detail

Also see https://sccoda.readthedocs.io/en/latest/getting_started.html

scCODA method: https://www.nature.com/articles/s41467-021-27150-6

### Package installation

The pertpy package must be installed in your environment (https://pypi.org/project/pertpy/):

module load miniforge

mamba activate scRNAseq

pip install 'pertpy[coda]'

Shutdown then restart the Jupyter notebook kernel and then import the below packages


In [ ]:
import warnings
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pertpy as pt
import scanpy as sc
import seaborn as sns
sc.settings.set_figure_params(scanpy = True, dpi=100, transparent=True, vector_friendly = True, frameon = False) 

### Inspect dataset

Load the dataset containing control and infected cells

In [ ]:
adata = pt.dt.haber_2017_regions()

Check number of cells and genes
- How do genes differ from the dataset above?

In [ ]:
adata

In [ ]:
adata.var_names

In [ ]:
adata.obs

Check which conditions are present

In [ ]:
set(list(adata.obs['condition']))

The effect of Heligmosomoides polygyrus and Salmonella infection conditions will be investigated on cell type abundance

### scCODA model

From https://www.sc-best-practices.org/conditions/compositional.html#compositional-analysis :

'scCODA belongs to the family of tools that require pre-defined clusters, most common cell types, to statistically derive changes in composition. Inspired by methods for compositional analysis of microbiome data, scCODA proposes a Bayesian approach to address the low replicate issue as commonly encountered in single-cell analysis[Büttner et al., 2021]. It models cell-type counts using a hierarchical Dirichlet-Multinomial model, which accounts for uncertainty in cell-type proportions and the negative correlative bias via joint modeling of all measured cell-type proportions. To ensure a uniquely identifiable solution and easy interpretability, the reference in scCODA is chosen to be a specific cell type. Hence, any detected compositional changes by scCODA always have to be viewed in relation to the selected reference.

However, scCODA assumes a log-linear relationship between covariates and cell abundance, which may not always reflect the underlying biological processes when using continuous covariates. A further limitation of scCODA is the inability to infer correlation structures among cell compositions beyond compositional effects. Furthermore, scCODA only models shifts in mean abundance, but does not detect changes in response variability[Büttner et al., 2021].

As a first step, we instantiate a scCODA model.

Then, we use load function to prepare a MuData object for subsequent processing, and it creates a compositional analysis dataset from the input adata. And we specify the cell_type_identifier as cell_label, sample_identifier as batch, and covariate_obs as condition in our case.'

In [ ]:
sccoda_model = pt.tl.Sccoda()

In [ ]:
sccoda_data = sccoda_model.load(
    adata,
    type="cell_level",
    generate_sample_level=True,
    cell_type_identifier="cell_label",
    sample_identifier="batch",
    covariate_obs=["condition"],
)
sccoda_data

We can use a boxplot overview of the cell type distributions across conditions

- What are the differences in the distributions of the cell types based on both conditions?
- How would we know whether these are statistically significant?

In [ ]:
sccoda_model.plot_boxplots(
    sccoda_data,
    modality_key="coda",
    feature_name="condition",
    figsize=(12, 5),
    add_dots=True,
 #   args_swarmplot={"palette": ["red"]},
)
plt.show()

Stacked boxplot to investigate cell type composition
- Does this confirm observations from above?

In [ ]:
sccoda_model.plot_stacked_barplot(
    sccoda_data, modality_key="coda", feature_name="condition", figsize=(4, 2)
)
plt.show()

'This visualization nicely displays the characteristics of compositional data: If we compare the Control and Salmonella groups, we can see that the proportion of Enterocytes greatly increases in the infected mice. Since the data is proportional, this leads to a decreased share of all other cell types to fulfill the sum-to-one constraint'

scCODA requires:
- anndata object
- formula - covariates, can be multiple
- reference cell, here as endocrine but could be set to 'automatic'

Why was endocrine chosen here?

In [ ]:
sccoda_data = sccoda_model.prepare(
    sccoda_data,
    modality_key="coda",
    formula="condition",
    reference_cell_type="Endocrine",
)
sccoda_model.run_nuts(sccoda_data, modality_key="coda", rng_key=1234)

Try this with different conditions

In [ ]:
sccoda_data["coda"].varm["effect_df_condition[T.Salmonella]"]

'The acceptance rate describes the fraction of proposed samples that are accepted after the initial burn-in phase, and can be an ad-hoc indicator for a bad optimization run. In the case of scCODA, the desired acceptance rate is between 0.4 and 0.9. Acceptance rates that are way higher or too low indicate issues with the sampling process.'

In [ ]:
sccoda_data

'scCODA selects credible effects based on their inclusion probability. The cutoff between credible and non-credible effects depends on the desired false discovery rate (FDR). A smaller FDR value will produce more conservative results, but might miss some effects, while a larger FDR value selects more effects at the cost of a larger number of false discoveries. The desired FDR level can be easily set after inference via sim_results.set_fdr(). Per default, the value is 0.05. Since, depending on the dataset, the FDR can have a major influence on the result, we recommend to try out different FDRs up to 0.2 to get the most prominent effects.'

- You can check how stringent 0.2 is compared to the default by testing different values below

In [ ]:
sccoda_model.set_fdr(sccoda_data, 0.2)

'To get the binary classification of compositional changes per cell type we use the credible_effects function of scCODA on the result object. Every cell type labeled as “True” is significantly more or less present. The fold-changes describe whether the cell type is more or less present. Hence, we will plot them alongside the binary classification below.'

- How may cell types are signficantly different between conditions?

In [ ]:
sccoda_model.credible_effects(sccoda_data, modality_key="coda")

 Plot the fold changes together with the binary classification

In [ ]:
sccoda_model.plot_effects_barplot(sccoda_data, modality_key = "coda", covariates = "condition")
plt.show()

- Does this agree with findings from the paper?

“After Salmonella infection, the frequency of mature enterocytes increased substantially.”[Haber et al., 2017]

“Heligmosomoides polygyrus caused an increase in the abundance of goblet and tuft cells.”[Haber et al., 2017]
